# VLANs and network segmentation  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 3.3.a: VLAN segmentation flow filter

**Skill:** Mitigate Risk · **Maps to:** 3.3.A · **Language:** Python 3 (pytest)

## Problem
A network switch enforces VLAN segmentation using the following default-deny policy:

- Traffic between two hosts **on the same VLAN** is **allowed**.
- Traffic from VLAN 20 (Employee) to VLAN 30 (Server) is **allowed** (business policy exception).
- All other cross-VLAN traffic is **blocked**.

Given a mapping of host names to VLAN IDs and a list of traffic flows, return the list of flows that are **blocked** by these rules.

## Function contract
Implement `solve(vlan_map: dict, flows: list) -> list` in the code editor (see the Reference section below).

## Input
- `vlan_map`: `dict[str, int]`, maps each host name to its VLAN ID (integer).
- `flows`: `list[tuple[str, str]]`, each tuple is `(src_host, dst_host)`.

## Output
A `list[tuple[str, str]]` of flows that are blocked, in the same order they appear in the input. Do not include allowed flows.

## Examples
```
vlan_map = {"WS-01": 20, "SRV-01": 30, "GUEST-01": 40, "MGMT-01": 10}
flows = [("WS-01","SRV-01"), ("GUEST-01","SRV-01"), ("WS-01","WS-01")]

solve(vlan_map, flows)
-> [("GUEST-01","SRV-01")]
# WS-01->SRV-01: VLAN 20->30, allowed by business exception
# GUEST-01->SRV-01: VLAN 40->30, blocked (no exception)
# WS-01->WS-01: same VLAN, allowed
```

## Constraints
- Deterministic, offline, standard library only.
- All host names in `flows` are guaranteed to be in `vlan_map`.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(vlan_map: dict, flows: list) -> list
```

### Input (`vlan_map`, `flows`)
- `vlan_map`: `dict[str, int]`, host name to VLAN ID.
- `flows`: `list[tuple[str, str]]`, list of (src_host, dst_host) pairs.

### Output (`result`)
`list[tuple[str, str]]`, the subset of `flows` that are **blocked**, preserving input order.

### Policy rules (applied in order)
1. If `vlan_map[src] == vlan_map[dst]`, **allowed** (same VLAN).
2. If `vlan_map[src] == 20` and `vlan_map[dst] == 30`, **allowed** (Employee -> Server exception).
3. All other cases, **blocked**.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- All hosts in `flows` are guaranteed to exist in `vlan_map`.
- Output list preserves the relative order of blocked flows from the input.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
"""Reference solution for 3.3.a. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests).
"""
from typing import List, Tuple, Dict


def solve(vlan_map: Dict[str, int], flows: List[Tuple[str, str]]) -> List[Tuple[str, str]]:
    """Determine which traffic flows are blocked by VLAN segmentation rules.

    Args:
        vlan_map: dict mapping host name to its VLAN ID.
        flows: list of tuples (src_host, dst_host).

    Returns:
        List of blocked flows in the same order.
    """
    blocked = []
    for src, dst in flows:
        v_src = vlan_map[src]
        v_dst = vlan_map[dst]
        # Rule 1: Same VLAN is allowed
        if v_src == v_dst:
            continue
        # Rule 2: VLAN 20 (Employee) to VLAN 30 (Server) is allowed
        if v_src == 20 and v_dst == 30:
            continue
        # Otherwise blocked
        blocked.append((src, dst))
    return blocked


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- visible cases -------------------------------------------------------
def test_example_1():
    vlan_map = {"WS-01": 20, "SRV-01": 30, "GUEST-01": 40, "MGMT-01": 10}
    flows = [("WS-01", "SRV-01"), ("GUEST-01", "SRV-01"), ("WS-01", "WS-01")]
    expected = [("GUEST-01", "SRV-01")]
    assert impl.solve(vlan_map, flows) == expected


def test_example_2():
    vlan_map = {"WS-01": 20, "SRV-01": 30, "GUEST-01": 40, "MGMT-01": 10}
    # Test one-way exception: 20 -> 30 is allowed, 30 -> 20 is blocked!
    flows = [("SRV-01", "WS-01"), ("MGMT-01", "WS-01")]
    expected = [("SRV-01", "WS-01"), ("MGMT-01", "WS-01")]
    assert impl.solve(vlan_map, flows) == expected


# ---- hidden / edge cases -------------------------------------------------
def test_edge_empty():
    vlan_map = {"WS-01": 20, "SRV-01": 30}
    assert impl.solve(vlan_map, []) == []


def test_all_blocked():
    vlan_map = {"WS-01": 20, "SRV-01": 30, "GUEST-01": 40}
    flows = [("GUEST-01", "WS-01"), ("SRV-01", "WS-01"), ("GUEST-01", "SRV-01")]
    assert impl.solve(vlan_map, flows) == flows


def test_all_allowed():
    vlan_map = {"WS-01": 20, "SRV-01": 30, "WS-02": 20}
    flows = [("WS-01", "WS-02"), ("WS-01", "SRV-01"), ("WS-02", "WS-01")]
    assert impl.solve(vlan_map, flows) == []


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
